In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

ROOT = Path(r"C:\Users\esarmiento\Documents\GitHub\emergency_access_peru")
PROCESSED_DIR = ROOT / "data" / "processed"

print("Cargando distritos...")
gdf_base = gpd.read_file(PROCESSED_DIR / "distritos_base.gpkg")
print("OK:", gdf_base.shape)

print("Cargando emergencias...")
df_emerg = pd.read_csv(PROCESSED_DIR / "emergencias_clean.csv", dtype={"ubigeo": str})
print("OK:", df_emerg.shape)

print("\nCargando centros poblados...")
gdf_cp_full = gpd.read_file(PROCESSED_DIR / "centros_poblados_distritales.gpkg")
df_cp = pd.DataFrame(gdf_cp_full.drop(columns="geometry"))
del gdf_cp_full
print("OK:", df_cp.shape)

print("\nCargando IPRESS con coords...")
df_ipress_coords = pd.read_csv(
    PROCESSED_DIR / "ipress_clean.csv",
    dtype={"ubigeo": str},
    usecols=["codigo_unico", "ubigeo", "norte", "este"]
).dropna(subset=["norte", "este"])
print("OK:", df_ipress_coords.shape)




Cargando distritos...
OK: (1873, 14)
Cargando emergencias...
OK: (977373, 14)

Cargando centros poblados...
OK: (135942, 8)

Cargando IPRESS con coords...
OK: (7941, 4)


# Metrics

Construye el índice de acceso a emergencias por distrito combinando tres componentes:

1. **Disponibilidad** — IPRESS por centro poblado en el distrito
2. **Actividad** — atenciones de emergencia por IPRESS en el distrito  
3. **Acceso espacial** — % de centros poblados con al menos un IPRESS a ≤ X km

Dos versiones:
- **Baseline**: umbral de acceso = 5 km
- **Alternativa**: umbral de acceso = 15 km

## Componente 1: Disponibilidad de IPRESS

Métrica: IPRESS por centro poblado en el distrito.
Un distrito con muchos IPRESS pero pocos centros poblados tiene alta disponibilidad.
Un distrito con pocos IPRESS y muchos centros poblados tiene baja disponibilidad.


In [ ]:
# Usamos n_ipress y n_centros_poblados que ya están en gdf_base
# Evitamos división por cero con replace(0, np.nan)
gdf_base["comp1_disponibilidad"] = (
    gdf_base["n_ipress"] / gdf_base["n_centros_poblados"].replace(0, np.nan)
)

print("Distritos sin centros poblados:", gdf_base["n_centros_poblados"].eq(0).sum())
print("Distritos sin IPRESS:", gdf_base["n_ipress"].eq(0).sum())
print("\nEstadísticas componente 1:")
print(gdf_base["comp1_disponibilidad"].describe())


Distritos sin centros poblados: 3
Distritos sin IPRESS: 14

Estadísticas componente 1:
count    1870.000000
mean        2.481639
std        19.971450
min         0.000000
25%         0.033898
50%         0.066667
75%         0.142857
max       365.000000
Name: comp1_disponibilidad, dtype: float64


## Componente 2: Actividad de emergencias

Métrica: total de atenciones de emergencia por IPRESS en el distrito (promedio 2022-2025).
Distritos con alta actividad por establecimiento tienen mayor carga de emergencias.


In [ ]:
# Agregamos atenciones totales por distrito (ignorando NaN de valores NE_XXXX)
emerg_por_dist = df_emerg.groupby("ubigeo").agg(
    total_atenciones=("nro_total_atenciones", "sum"),
    total_atendidos=("nro_total_atendidos", "sum"),
    n_ipress_reportantes=("co_ipress", "nunique")
).reset_index()

# Atenciones por IPRESS reportante en el distrito
emerg_por_dist["comp2_actividad"] = (
    emerg_por_dist["total_atenciones"] / emerg_por_dist["n_ipress_reportantes"]
)

print("Distritos con datos de emergencia:", len(emerg_por_dist))
print("\nEstadísticas componente 2:")
print(emerg_por_dist["comp2_actividad"].describe())


Distritos con datos de emergencia: 1190

Estadísticas componente 2:
count      1190.000000
mean       7970.734726
std       31289.443707
min           0.000000
25%           0.000000
50%           2.450000
75%        1856.787946
max      620386.000000
Name: comp2_actividad, dtype: float64


## Componente 3: Acceso espacial

Métrica: % de centros poblados del distrito que tienen al menos un IPRESS a ≤ X km.

- **Baseline**: X = 5 km
- **Alternativa**: X = 15 km

Usamos distancia euclidiana en grados convertida a km aproximados (1° ≈ 111 km en Perú).


In [ ]:
from scipy.spatial import cKDTree

def calcular_acceso_cp(df_cp, df_ipress_coords, umbral_km):
    """
    Para cada centro poblado calcula si tiene al menos un IPRESS
    dentro del umbral de distancia (en km). Retorna % por distrito.
    """
    umbral_grados = umbral_km / 111.0  # conversión aproximada km → grados

    # Árbol de búsqueda sobre coordenadas de IPRESS
    ipress_coords = df_ipress_coords[["norte", "este"]].values
    tree = cKDTree(ipress_coords)

    # Centros poblados con coordenadas válidas
    cp_valid = df_cp.dropna(subset=["x", "y", "ubigeo"]).copy()
    cp_coords = cp_valid[["x", "y"]].values

    # Distancia al IPRESS más cercano para cada centro poblado
    distancias, _ = tree.query(cp_coords, k=1)
    cp_valid["tiene_acceso"] = distancias <= umbral_grados

    # % de CP con acceso por distrito
    acceso_dist = cp_valid.groupby("ubigeo").agg(
        n_cp=("tiene_acceso", "count"),
        n_cp_con_acceso=("tiene_acceso", "sum")
    ).reset_index()
    acceso_dist["pct_acceso"] = acceso_dist["n_cp_con_acceso"] / acceso_dist["n_cp"]

    return acceso_dist

# Calculamos para ambos umbrales
print("Calculando baseline (5 km)...")
acceso_5km  = calcular_acceso_cp(df_cp, df_ipress_coords, umbral_km=5)
print("Calculando alternativa (15 km)...")
acceso_15km = calcular_acceso_cp(df_cp, df_ipress_coords, umbral_km=15)

print("\nBaseline 5km — % acceso promedio:", acceso_5km["pct_acceso"].mean().round(3))
print("Alternativa 15km — % acceso promedio:", acceso_15km["pct_acceso"].mean().round(3))


Calculando baseline (5 km)...
Calculando alternativa (15 km)...

Baseline 5km — % acceso promedio: 0.313
Alternativa 15km — % acceso promedio: 0.432


## Construcción del índice final

Los tres componentes se normalizan de 0 a 1 con min-max y se promedian.
Un score más alto indica mejor acceso a emergencias.


In [ ]:
def minmax(series):
    """Normaliza una serie al rango [0, 1]."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0
    return (series - mn) / (mx - mn)

def construir_indice(gdf_base, emerg_por_dist, acceso_dist, sufijo):
    """Construye el índice de acceso para un umbral dado."""
    df = gdf_base[["ubigeo", "distrito", "departamen", "provincia",
                   "n_ipress", "n_centros_poblados", "comp1_disponibilidad"]].copy()

    # Unimos componente 2
    df = df.merge(emerg_por_dist[["ubigeo", "comp2_actividad"]], on="ubigeo", how="left")
    df["comp2_actividad"] = df["comp2_actividad"].fillna(0)

    # Unimos componente 3
    df = df.merge(acceso_dist[["ubigeo", "pct_acceso"]], on="ubigeo", how="left")
    df["pct_acceso"] = df["pct_acceso"].fillna(0)
    df = df.rename(columns={"pct_acceso": "comp3_acceso"})

    # Normalizamos los tres componentes
    df["comp1_norm"] = minmax(df["comp1_disponibilidad"].fillna(0))
    df["comp2_norm"] = minmax(df["comp2_actividad"])
    df["comp3_norm"] = minmax(df["comp3_acceso"])

    # Score final: promedio igualitario de los tres componentes
    df[f"score_{sufijo}"] = (df["comp1_norm"] + df["comp2_norm"] + df["comp3_norm"]) / 3

    return df

# Construimos baseline (5km) y alternativa (15km)
df_baseline   = construir_indice(gdf_base, emerg_por_dist, acceso_5km,  "baseline")
df_alternativa = construir_indice(gdf_base, emerg_por_dist, acceso_15km, "alternativa")

print("Score baseline — estadísticas:")
print(df_baseline["score_baseline"].describe().round(3))
print("\nScore alternativa — estadísticas:")
print(df_alternativa["score_alternativa"].describe().round(3))


Score baseline — estadísticas:
count    1873.000
mean        0.111
std         0.133
min         0.000
25%         0.000
50%         0.012
75%         0.238
max         0.721
Name: score_baseline, dtype: float64

Score alternativa — estadísticas:
count    1873.000
mean        0.151
std         0.161
min         0.000
25%         0.000
50%         0.016
75%         0.333
max         0.721
Name: score_alternativa, dtype: float64


In [ ]:
# Unimos ambos scores para comparar
df_comparacion = df_baseline[["ubigeo", "distrito", "departamen", "score_baseline"]].merge(
    df_alternativa[["ubigeo", "score_alternativa"]], on="ubigeo"
)

# Diferencia entre alternativa y baseline
df_comparacion["diferencia"] = df_comparacion["score_alternativa"] - df_comparacion["score_baseline"]

# Distritos que más cambian al ampliar el umbral de 5 a 15km
print("Distritos que más mejoran con umbral 15km:")
print(df_comparacion.nlargest(5, "diferencia")[["distrito", "departamen", "score_baseline", "score_alternativa", "diferencia"]].to_string())

print("\nDistritos que no cambian (diferencia = 0):")
print((df_comparacion["diferencia"] == 0).sum())


Distritos que más mejoran con umbral 15km:
         distrito departamen  score_baseline  score_alternativa  diferencia
775         SAYLA   AREQUIPA        0.000038           0.333371    0.333333
287   SUYCKUTAMBO      CUSCO        0.040569           0.322756    0.282187
1259     LA OROYA      JUNIN        0.052523           0.333475    0.280952
773        TAURIA   AREQUIPA        0.000035           0.269266    0.269231
488      ORONCCOY   AYACUCHO        0.064815           0.333333    0.268519

Distritos que no cambian (diferencia = 0):
1191


In [ ]:
# Clasificamos distritos en terciles según el score baseline
df_baseline["clasificacion"] = pd.qcut(
    df_baseline["score_baseline"],
    q=3,
    labels=["Subatendido", "Acceso medio", "Mejor atendido"]
)

print("Distribución de clasificación:")
print(df_baseline["clasificacion"].value_counts())

# Top 10 mejor atendidos
print("\nTop 10 distritos mejor atendidos:")
print(df_baseline.nlargest(10, "score_baseline")[
    ["distrito", "departamen", "score_baseline", "clasificacion"]
].to_string())

# Top 10 más subatendidos (con al menos 1 CP)
print("\nTop 10 distritos más subatendidos:")
print(df_baseline[df_baseline["n_centros_poblados"] > 0].nsmallest(10, "score_baseline")[
    ["distrito", "departamen", "score_baseline", "clasificacion"]
].to_string())


NameError: name 'pd' is not defined